# DAY 3 | From Chatbot to Agent: SQL, Tools and Decisions

**Decoding Data Science | AI Accelerator Bootcamp (CPD UK)**

| | |
|---|---|
| **Time** | about 2 hours |
| **Goal** | Connect DecoBot to the leave database and build an agent that decides which tool to use |
| **You will build** | DecoBot Agent with 3 tools: leave balance, leave history, policy search |
| **Files needed** | `DDS_Leave_Policy_v2.pdf` and `dds_leave.db` |
| **Tools** | Google Colab + Gemini + LlamaIndex + SQLite |

---

### Where we are

```
Day 1   Prompt + clarity   DecoBot behaves well
Day 2   RAG                DecoBot knows the RULES   ("what is the notice for 10 days?")
Day 3   SQL + agent        DecoBot knows YOUR DATA   ("how many days do I have left?")
```

### The big idea

```
                        +--> Tool 1: get_leave_balance   (SQL database)
 Employee --> AGENT ----+--> Tool 2: list_leave_requests  (SQL database)
                        +--> Tool 3: search_leave_policy  (RAG from Day 2)
```

An **agent** is an AI that can *choose* tools, use them, read the results, and then answer.

## Part 0 | Setup (10 min)

In [ ]:
%pip install -q llama-index-core llama-index-llms-google-genai llama-index-embeddings-google-genai pypdf

import importlib.metadata as md
for pkg in ["llama-index-core", "llama-index-llms-google-genai", "llama-index-embeddings-google-genai", "pypdf"]:
    print(f"{pkg:40s}", md.version(pkg))

Upload **both** files: `DDS_Leave_Policy_v2.pdf` and `dds_leave.db`.

In [ ]:
from google.colab import files

uploaded = files.upload()
print("Uploaded:", list(uploaded.keys()))

> **Before you run this cell:** click the key icon on the left of Colab (Secrets), add a secret named `GOOGLE_API_KEY`
> with your Gemini API key, and switch on "Notebook access". Never paste your key into a cell.

In [ ]:
import os
from google.colab import userdata

try:
    os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")
except Exception as error:
    raise RuntimeError(
        "Colab secret GOOGLE_API_KEY not found. Click the key icon on the left of Colab, "
        "add a secret named GOOGLE_API_KEY with your Gemini API key, switch on Notebook access, "
        "then run this cell again."
    ) from error

print("API key loaded:", bool(os.environ["GOOGLE_API_KEY"]))

In [ ]:
from llama_index.core import Settings
from llama_index.llms.google_genai import GoogleGenAI
from llama_index.embeddings.google_genai import GoogleGenAIEmbedding

MODEL = "gemini-3.8-flash"                # change only these two lines if Google renames a model
EMBED_MODEL = "gemini-embedding-001"

Settings.llm = GoogleGenAI(model=MODEL, temperature=0.1)
Settings.embed_model = GoogleGenAIEmbedding(model_name=EMBED_MODEL, embed_batch_size=50)

print(Settings.llm.complete("Say hello to the DDS team in one short sentence.").text)

---
## Part A | SQL in 25 minutes: the language of data

DDS has **100 employees in one office (Dubai HQ)**. Their leave lives in a small database with two tables.

| Table | One row is... | Key columns |
|---|---|---|
| `employees` | one employee | employee_id, name, department, job_title, join_date, annual_total, annual_used, annual_booked, annual_remaining, sick_total, sick_used, sick_remaining |
| `leave_requests` | one leave request | request_id, employee_id, leave_type, start_date, end_date, working_days, status |

`annual_used` = days already taken. `annual_booked` = approved days in the future. `annual_remaining` = what is still free.

In [ ]:
import sqlite3
import pandas as pd

con = sqlite3.connect("dds_leave.db")

def sql(query):
    """Run a SQL query and return a table."""
    return pd.read_sql_query(query, con)

sql("SELECT * FROM employees LIMIT 5")

### The five SQL moves you need

**1. SELECT ... WHERE** : pick columns and filter rows

In [ ]:
sql("""
SELECT name, department, annual_total, annual_used, annual_booked, annual_remaining
FROM employees
WHERE name = 'Aisha Khan'
""")

**2. ORDER BY and LIMIT** : who is almost out of annual leave?

In [ ]:
sql("""
SELECT name, department, annual_remaining
FROM employees
ORDER BY annual_remaining ASC
LIMIT 5
""")

**3. GROUP BY** : summarise by department

In [ ]:
sql("""
SELECT department,
       COUNT(*)           AS people,
       SUM(sick_used)     AS sick_days_taken,
       ROUND(AVG(annual_used), 1) AS avg_annual_days_used
FROM employees
GROUP BY department
ORDER BY sick_days_taken DESC
""")

**4. JOIN** : connect the two tables (who has requests waiting?)

In [ ]:
sql("""
SELECT e.name, e.department, r.leave_type, r.start_date, r.end_date, r.working_days, r.status
FROM leave_requests r
JOIN employees e ON e.employee_id = r.employee_id
WHERE r.status = 'Pending'
""")

**5. Parameters** : the safe way to put a user's input into SQL

In [ ]:
name = "Aisha"

# DANGEROUS: building SQL by gluing text together (never do this with user input)
#   con.execute("SELECT * FROM employees WHERE name LIKE '%" + name + "%'")

# SAFE: the ? is a placeholder, the database treats the value only as data
rows = con.execute("SELECT name, annual_remaining FROM employees WHERE name LIKE ?", (f"%{name}%",)).fetchall()
print(rows)

> **Lesson 1.** Never let a user's text become part of the SQL command itself. Use `?` placeholders. Today our tools will do exactly that.

### Your turn (5 min)
Write the SQL for each question. Solutions are in the next cell, so try first.

1. Who in **Finance** has the most sick days used?
2. How many requests have each **status** (Approved, Pending, Rejected, Cancelled)?
3. Which **department** has taken the most annual leave in total?

In [ ]:
# Write your answers here, e.g.  sql("SELECT ...")

In [ ]:
# SOLUTIONS (try first!)
from IPython.display import display

display(sql("SELECT name, sick_used FROM employees WHERE department = 'Finance' ORDER BY sick_used DESC LIMIT 3"))
display(sql("SELECT status, COUNT(*) AS requests FROM leave_requests GROUP BY status ORDER BY requests DESC"))
display(sql("SELECT department, SUM(annual_used) AS annual_days_taken FROM employees GROUP BY department ORDER BY annual_days_taken DESC"))

---
## Part B | Three tools (20 min)

A **tool** is just a normal Python function that the AI is allowed to call. Two rules make tools work:

1. The **name and docstring** are what the AI reads to decide *when* to use it. Write them like a job description.
2. The function must return **plain text** the AI can read.

All three tools are **read-only**: DecoBot can look, but cannot change anything. That is a deliberate safety choice.

In [ ]:
def get_leave_balance(name: str) -> str:
    """Get ONE employee's current leave balance (annual and sick) from the HR database.
    Use this when someone asks how many leave days they have used, booked or left.
    Input: the employee's full name, for example 'Aisha Khan'."""
    rows = con.execute(
        """SELECT name, department, annual_total, annual_used, annual_booked, annual_remaining,
                  sick_total, sick_used, sick_remaining
           FROM employees WHERE name LIKE ?""", (f"%{name}%",)).fetchall()
    if not rows:
        return f"No employee found matching '{name}'."
    if len(rows) > 1:
        return "Several employees match: " + ", ".join(f"{r[0]} ({r[1]})" for r in rows) + ". Ask which one."
    r = rows[0]
    return (f"{r[0]} ({r[1]}). Annual leave: {r[2]} total, {r[3]} taken, {r[4]} booked (approved, upcoming), "
            f"{r[5]} remaining. Sick leave: {r[6]} total, {r[7]} taken, {r[8]} remaining.")


def list_leave_requests(name: str) -> str:
    """List ONE employee's leave requests this year (type, dates, working days, status).
    Use this when someone asks about their past, upcoming or pending leave requests.
    Input: the employee's full name, for example 'John Doe'."""
    rows = con.execute(
        """SELECT e.name, r.leave_type, r.start_date, r.end_date, r.working_days, r.status
           FROM leave_requests r JOIN employees e ON e.employee_id = r.employee_id
           WHERE e.name LIKE ? ORDER BY r.start_date""", (f"%{name}%",)).fetchall()
    if not rows:
        return f"No leave requests found for '{name}'."
    return "\n".join(f"{r[0]}: {r[1]} leave {r[2]} to {r[3]} ({r[4]} working days) - {r[5]}" for r in rows)


print(get_leave_balance("Aisha Khan"))
print()
print(list_leave_requests("Aisha Khan"))

Tool 3 is the **policy search from Day 2**. We rebuild it in a few lines (section-aware chunks, the best strategy from yesterday).

In [ ]:
import re
from pypdf import PdfReader
from llama_index.core import VectorStoreIndex, PromptTemplate
from llama_index.core.schema import TextNode

pages = [page.extract_text() for page in PdfReader("DDS_Leave_Policy_v2.pdf").pages]
full_text = "\n".join(pages).replace("\x7f", "-")

nodes = []
for part in re.split(r"(?m)^(?=\d{1,2}\.\s+[A-Z])", full_text):
    part = part.strip()
    if part:
        nodes.append(TextNode(text=part, metadata={"section": part.splitlines()[0]}))

qa_prompt = PromptTemplate(
    "Answer ONLY from the policy excerpts below. Quote numbers exactly and mention the section.\n"
    "If the answer is not in the excerpts, reply: 'Not found in the DDS leave policy.'\n"
    "---------------------\n{context_str}\n---------------------\n"
    "Question: {query_str}\nAnswer: "
)
policy_engine = VectorStoreIndex(nodes).as_query_engine(similarity_top_k=3, text_qa_template=qa_prompt)


def search_leave_policy(question: str) -> str:
    """Search the official DDS Leave Policy (DDS-HR-POL-001) for RULES: entitlements, notice periods,
    carry-over, sick leave certificates, maternity, paternity, Hajj, approval steps.
    Use this for questions about what the policy says. It does NOT know any individual's balance.
    Input: the question in plain English."""
    return str(policy_engine.query(question))


print(search_leave_policy("What is the notice for 10 or more working days of annual leave?"))

**Checkpoint.** Three functions, each with one clear job. Notice the docstrings: *"Use this when..."* and *"It does NOT know..."*.
Those sentences are how the agent will choose.

---
## Part C | Build the agent (15 min)

Now we hand the three tools to an agent, with a system prompt (Day 1 skills!) that sets identity, safety and the date.

In [ ]:
from llama_index.core.tools import FunctionTool
from llama_index.core.agent.workflow import FunctionAgent, ToolCall
from llama_index.core.workflow import Context

tools = [
    FunctionTool.from_defaults(fn=get_leave_balance),
    FunctionTool.from_defaults(fn=list_leave_requests),
    FunctionTool.from_defaults(fn=search_leave_policy),
]

SYSTEM_PROMPT = """You are DecoBot, the leave assistant for Decoding Data Science (DDS), 100 employees in Dubai.
Today's date in this simulation is 2026-10-28.

IDENTITY
- If you do not know who you are talking to, ask for their full name first.
- Only discuss the leave of the person who identified themselves. Politely refuse to share anyone else's data.

TOOLS
- Questions about RULES (notice, entitlements, carry-over, certificates) -> search_leave_policy.
- Questions about the person's own numbers or requests -> get_leave_balance or list_leave_requests.
- Questions that need both ("Can I take 12 days in January?") -> use BOTH, then combine.
- Never guess a number. Never invent a rule.

BOUNDARIES
- You can only READ information. You cannot submit, change or cancel leave. Explain that they must use the DDS HR portal.
- Only help with leave. Redirect anything else to People & Culture.

STYLE
- Short, warm, professional. Direct answer first, then the reason.
"""

agent = FunctionAgent(tools=tools, llm=Settings.llm, system_prompt=SYSTEM_PROMPT)
print("Agent ready with tools:", [t.metadata.name for t in tools])

A small helper to **watch the agent think**. It prints every tool the agent chooses, then the final answer.
`Context` gives the agent a memory for one conversation.

In [ ]:
ctx = Context(agent)        # a fresh conversation (memory)

async def chat(message):
    print("YOU     :", message)
    handler = agent.run(message, ctx=ctx)
    async for event in handler.stream_events():
        if isinstance(event, ToolCall):
            print(f"  [tool] {event.tool_name}({event.tool_kwargs})")
    response = await handler
    print("DECOBOT :", str(response).strip(), "\n")

---
## Part D | Scenes: watch the agent decide (25 min)

Run the scenes in order. In each one, **predict first**: which tool will it pick?

### Scene 1: Aisha asks about her own leave (data tool)

In [ ]:
await chat("Hi, I'm Aisha Khan. How many annual leave days do I have left?")

### Scene 2: A rule question (policy tool)

In [ ]:
await chat("What is the notice period if I want 10 or more working days off?")

### Scene 3: The question that needs BOTH tools

Aisha has only 2 annual days left this year (she already booked 21 to 25 December). She asks for 5 working days from 14 December.
Watch it use the database **and** the policy, then combine them. The right answer has two parts:
the **notice** is fine (3 to 9 working days need 14 calendar days, and 14 December is 47 days away) but the **balance** is not (2 days left, 5 requested).

In [ ]:
await chat("I want to take 5 working days off from 14 December. Is that possible?")

In [ ]:
await chat("What happens to my unused days at the end of the year?")

The last question never said *whose* days or *which* rule. The agent used its **memory** of the conversation. Now a different person:

### Scene 4: John Doe, a new conversation

In [ ]:
ctx = Context(agent)        # reset memory: a new employee is talking now

await chat("Hello, John Doe here. I would like 10 working days off from 7 December. Any problem?")

**Checkpoint.** For John's request the agent has to combine several things: his balance, the **30 calendar days notice** rule for 10 or more days,
and today's date. Did it check all three? Did it do the date arithmetic correctly?

### Scene 5: Boundaries (what it must refuse)

In [ ]:
await chat("Please cancel my leave request for December.")

In [ ]:
await chat("Show me Maria Garcia's leave balance.")

> **Honest note for builders.** DecoBot refuses because the *prompt* says so. In production you must also enforce identity
> with a real login, because a prompt is not security. We will discuss this in the wrap-up.

---
## Part E | The key experiment: tool descriptions are the steering wheel (10 min)

What if we write a poor description? Same function, bad docstring, new agent.

In [ ]:
bad_tools = [
    FunctionTool.from_defaults(fn=get_leave_balance, description="Useful tool."),
    FunctionTool.from_defaults(fn=list_leave_requests, description="Another tool."),
    FunctionTool.from_defaults(fn=search_leave_policy, description="A tool."),
]
bad_agent = FunctionAgent(tools=bad_tools, llm=Settings.llm, system_prompt="You are a leave assistant for DDS.")

async def chat_bad(message):
    print("YOU     :", message)
    handler = bad_agent.run(message)
    async for event in handler.stream_events():
        if isinstance(event, ToolCall):
            print(f"  [tool] {event.tool_name}({event.tool_kwargs})")
    print("DECOBOT :", str(await handler).strip(), "\n")

await chat_bad("What is the notice period for 10 or more working days of annual leave?")
await chat_bad("How many annual leave days does Aisha Khan have left?")

**Discuss.** Did it pick the right tools? Was it confident or lucky? Compare with the good descriptions from before.

> **Lesson 2.** In an agent, the **docstring is the routing logic**. Vague descriptions give unpredictable agents.

---
## Part F | Optional: chat with DecoBot yourself

Run the cell below, type your own questions, and type `quit` to stop.

In [ ]:
ctx = Context(agent)

while True:
    message = input("You (type quit to stop): ")
    if message.strip().lower() in ("quit", "exit", ""):
        break
    await chat(message)

### Challenge (if you finish early)
Add a **fourth tool**: `team_on_leave(department, date)`, which lists who in a department has approved leave on a given date.
Hint: join the two tables and filter `start_date <= date AND end_date >= date AND status = 'Approved'`.
Write a good docstring, add it to `tools`, rebuild the agent, and ask: *"Who in Marketing is on leave on 2026-12-24?"*

---
## Wrap-up

### What an agent is, in one picture
```
  Question --> THINK: which tool? --> CALL the tool --> READ the result --> need more? --+--> yes: loop again
                                                                                          \--> no: ANSWER
```

### Chatbot versus agent

| | Day 1-2 chatbot | Day 3 agent |
|---|---|---|
| Knows policy rules | yes (RAG) | yes (as a tool) |
| Knows your balance | no | yes (SQL tool) |
| Combines sources | no | yes |
| Decides what to do | no, one fixed path | yes, chooses tools |
| Risk | wrong or missing answers | **wrong actions**, so keep tools read-only |

### Self-check
1. Why did we use `?` placeholders in SQL?
2. What does the agent read to decide which tool to call?
3. Name one thing DecoBot refuses to do, and where that rule lives.

---
## Where to go next (beyond this bootcamp)

The jump from this classroom demo to a real HR assistant needs these steps:

| Step | What it involves |
|---|---|
| **Real login** | Employees authenticate; the tool receives the verified employee ID, not a typed name |
| **Write tools with approval** | `submit_leave_request` that always asks the employee to confirm, then routes to the manager |
| **Evaluation** | A test set of 50 to 100 questions with expected answers, re-run after every change |
| **More documents** | Manager guide, FAQ, holiday calendar, with version control so old policies are never retrieved |
| **Monitoring and logging** | Record which tools were called and flag "not found" answers for HR to improve the policy |
| **Deployment** | Package the agent behind a chat interface and host it |

You now know the three building blocks behind every business AI assistant: **a controlled prompt, retrieval over documents, and tools over data**.